In [4]:
import tiktoken
from datasets import load_dataset
import re
from collections import Counter

In [5]:
storiesforthewin = load_dataset("Sabijn/StoriesfortheWin", split="train")

In [6]:
babybabelNL = load_dataset("BabyLM-community/babylm-nld", split="train")

In [7]:
def count_words(dataset):
	total_words = 0
	vocab = Counter()

	for example in dataset:
		tokens = example["tokens"]
		total_words += len(tokens)
		vocab.update(tokens)

	unique_words = len(vocab)

	return total_words, unique_words, vocab

In [8]:
def dataset_stats(dataset, text_column='text'):
	vocab = Counter()
	total_words = 0

	for example in dataset:
		tokens = re.findall(r"\b\w+\b", example[text_column].lower())
		total_words += len(tokens)
		vocab.update(tokens)

	unique_words = len(vocab)

	return total_words, unique_words, vocab.most_common(20)

In [9]:
# Tinystories 450M / 250K
# BabyLM English 100M / 1.7M
# StoriesfortheWin 92M (148M tokens) / 230K

storiesforthewin_stats = dataset_stats(storiesforthewin, text_column='story')

print("Total words:", storiesforthewin_stats[0])
print("Unique words:", storiesforthewin_stats[1])
print("20 most common", storiesforthewin_stats[2])

Total words: 92192259
Unique words: 234022
20 most common [('de', 4269781), ('een', 3403920), ('en', 2926890), ('hij', 2692614), ('het', 2014785), ('was', 1933707), ('ze', 1760497), ('te', 1457523), ('naar', 1320921), ('dat', 1294975), ('in', 1253679), ('zijn', 1199749), ('maar', 1167618), ('er', 1077041), ('met', 1056532), ('ik', 1026990), ('die', 940351), ('om', 934547), ('van', 930096), ('op', 897410)]


In [10]:
# BabybabelNL 97M / 626K

babybabelNL_stats = dataset_stats(babybabelNL, 'text')

print("Total words:", babybabelNL_stats[0])
print("Unique words:", babybabelNL_stats[1])
print("20 most common", babybabelNL_stats[2])

Total words: 97118965
Unique words: 626765
20 most common [('de', 3637985), ('je', 2581296), ('het', 2492602), ('ik', 2358590), ('een', 2083664), ('is', 1779302), ('en', 1702119), ('dat', 1587038), ('van', 1560802), ('in', 1374573), ('niet', 1238506), ('op', 903897), ('zijn', 806774), ('te', 772298), ('wat', 753912), ('ze', 729196), ('met', 720216), ('die', 686791), ('voor', 676693), ('we', 672972)]


In [11]:
# child-books babybabelNL 4.5M / 77K
babybabelNL_books = babybabelNL.filter(lambda example: example['category'] == 'child-books')

babybabelbooks_stats = dataset_stats(babybabelNL_books, 'text')

print("Total words:", babybabelbooks_stats[0])
print("Unique words:", babybabelbooks_stats[1])
print("20 most common", babybabelbooks_stats[2])

Total words: 4582650
Unique words: 77668
20 most common [('de', 184256), ('en', 115312), ('het', 114457), ('een', 107833), ('ik', 90128), ('ze', 75981), ('hij', 72473), ('dat', 68143), ('van', 62398), ('in', 61134), ('op', 54961), ('zijn', 53648), ('je', 52510), ('is', 50812), ('niet', 50625), ('maar', 44253), ('met', 40085), ('te', 38371), ('naar', 34748), ('die', 34746)]


In [12]:
print('Ratio unique / total words')
print('Babybabelbooks', babybabelbooks_stats[1] / babybabelbooks_stats[0])
print('BabybabelNL', babybabelNL_stats[1] / babybabelNL_stats[0])
print('StoriesfortheWin', storiesforthewin_stats[1] / storiesforthewin_stats[0])
print('TinyStories', 250_000 / 450_000_000)
print('BabyLM', 1_700_000 / 100_000_000)

Ratio unique / total words
Babybabelbooks 0.016948272287868372
BabybabelNL 0.006453579895543573
StoriesfortheWin 0.0025384126882062844
TinyStories 0.0005555555555555556
BabyLM 0.017


In [13]:
def load_tokenizer(special_tokens):
	enc = tiktoken.get_encoding("cl100k_base")
	
	enc = tiktoken.Encoding(
		name="cl100k_custom",
		pat_str=enc._pat_str,
		mergeable_ranks=enc._mergeable_ranks,
		special_tokens={**enc._special_tokens, **special_tokens},
	)

	return enc

BOS = "<|begin_of_text|>"
EOS = "<|endoftext|>"
special_tokens = {
	BOS: 100264,
	EOS: 100257,
}

tokenizer = load_tokenizer(special_tokens)

In [14]:
def tokenize_function(examples, tokenizer, bos_token, eos_token):
    return {
        "input_ids": [
            tokenizer.encode(text, allowed_special={bos_token, eos_token})
            for text in examples["story"]
        ]
    }

In [15]:
# tokenize
tokenized_storiesforthewin = storiesforthewin.map(
	lambda batch: tokenize_function(batch, tokenizer, BOS, EOS),
	batched=True,
	remove_columns=storiesforthewin.column_names,
	desc="Tokenizing train dataset",
)

In [16]:
from tqdm import tqdm
total_tokens = 0
for row in tqdm(tokenized_storiesforthewin):
    total_tokens += len(row['input_ids'])
print(total_tokens)

100%|██████████| 680300/680300 [00:28<00:00, 24160.49it/s]

148436165


In [18]:
babybabel_without_books = babybabelNL.filter(lambda example: example['category'] != 'child-books')
babybabelbooks_stats = dataset_stats(babybabel_without_books, 'text')

print("Total words:", babybabelbooks_stats[0])
print("Unique words:", babybabelbooks_stats[1])
print("20 most common", babybabelbooks_stats[2])

Total words: 92536315
Unique words: 610615
20 most common [('de', 3453729), ('je', 2528786), ('het', 2378145), ('ik', 2268462), ('een', 1975831), ('is', 1728490), ('en', 1586807), ('dat', 1518895), ('van', 1498404), ('in', 1313439), ('niet', 1187881), ('op', 848936), ('zijn', 753126), ('te', 733927), ('wat', 728232), ('met', 680131), ('ze', 653215), ('die', 652045), ('voor', 651734), ('we', 648821)]
